# Structural annotation course

This tutorial will guide you through the process of creating an structural annotation using a reference genome and long reads RNA data from a PacBio sequencing experiment. 

<details>
<summary> 📖 Theoretical background</summary>

The first step in understanding a genome typically involves **structural annotation**—the process of identifying protein-coding genes and their associated features. One core method used in this phase is **_ab initio_ gene prediction**, which relies solely on the genomic sequence itself. This approach uses statistical models, such as **Hidden Markov Models (HMMs)**, trained to detect signal sensors—including **splice sites**, **start codons**, and **stop codons**-as well as content sensors, like **codon usage patterns** that are characteristic of coding regions.

While **_ab initio_ prediction** is valuable because it does not require prior experimental data, it often struggles with **lower accuracy**, especially in capturing **complete gene structures**, **untranslated regions (UTRs)**, and **alternative isoforms**. Moreover, its effectiveness is heavily dependent on the availability of **species-specific training models**.

To address these limitations, **evidence-based genome annotation** combines experimental data with _ab initio_ predictions to enhance accuracy. Among the most powerful sources of evidence is **long-read RNA sequencing (lr-RNA seq)**. Unlike traditional short-read RNA-seq, lr-RNA seq can capture **full-length transcripts**, offering direct insights into exon-intron boundaries, alternative splicing, transcription start and end sites, and UTRs.

When used as extrinsic evidence or **"hints"**, lr-RNA seq data can dramatically improve the performance of _ab initio_ gene finders like AUGUSTUS. This integration results in **more accurate and complete gene models**, enabling the identification of **novel isoforms** and providing a deeper understanding of the transcriptome. Such an approach is especially valuable for the **annotation of non-model organisms**, where genomic resources are often limited.

</details><br>

We will use different software in our process of generating the three target annotations and the consequent benchmark. These are:

1. **BUSCO** to generate the benchmark of the resulting annotations
2. **AUGUSTUS** to generate the gene model and both annotations
3. **AGAT** to extract the proteome from the annotation
4. **cd-hit** will clusters the high-confidence genes to eliminate redundancy
5. **IsoQuant** to produce the lr-RNA seq transcriptome from the raw lr-RNA seq data
6. **OMARk** to assess the completeness and consistency of the proteome
7. **SQANTI3** to filter the raw transcriptome and eliminate low quality isoforms

## Table of contents

- [0. Prerequisites](#0-prerequisites)
  - [Software installation](#software-installation)
  - [Data download](#data-download)
- [1. Gene model training](#1-gene-model-training)
  - [1.1 Preprocessing](#11-preprocessing)  
  - [1.2 Gene model creation](#12-gene-model-creation)
  - [1.3 Augustus training](#13-augustus-training)
- [2. _Ab initio_ prediction](#2-ab-initio-prediction)
  - [2.1 Quality Control](#21-quality-control)
- [3. _Evidence driven_ annotation](#3-evidence-driven-annotation)
  - [3.1 Preprocessing of lr-RNA seq data](#31-preprocessing-of-lr-rna-seq-data)
  - [3.2 Hint creation](#32-hint-creation)
  - [3.3 Final evidence-driven annotation](#33-final-evidence-driven-annotation)



# 0. Prerequisites 

Before starting the tutorial, it is key to have a clean and organized working environment. The initial step, even before processing any data is to prepare the working environment. In bioinformatics, an organized workspace is vital, so when you come after some time to your project, you can find and understand what you were doing, rather than spend hours searching through weirdly named directories. It is important to always create three directories:


- scripts: all the scripts will be stored here, with meaningful names
- data: Raw data will go in here and, if you want and need, databases
- results: Create a sub directory for every different process you do. If you run a process multiple times with different parameters, include them in the directory name, so you will differentiate them in the future.


In [11]:
%%bash
mkdir -p results/ab_initio
mkdir -p results/evidence_based
mkdir -p results/evidence_driven
mkdir -p results/quality_control


### Software installation

All of the software that we will use today can be installed directly using one single conda environment and command. This is done for the sake of simplicity. However, some of the software that will be used today has their own recomendations for installation. For example, Busco recommends to do a manual installation, due to some versioning conflicts that might arise with accesory software. Always remember to read and follow the guidelines of the developers.

_All the needed software has been preinstalled for you to use in the conda environment "augustus"_

### Data download

For the tutorial, we will use the chromosome 8 of the human genome, following the SQANTI3 practical we have done. The transcriptome and the reference will be the same that in that practical, to optimize resources. However, in your _data_ directories you will have all of the processed files for you to run in case anything went wrong in the preivous practices. 

# 1. *Evidence-based* annotation

To begin with, we will generate the first and most simple annotation posible. We will just need a reconstructed transcriptome. For the sake of simplicity, the reads have already been reconstructed into a transcriptome using IsoQuant. Yes, IsoQuant, not IsoTools, as we did before. Why? You might ask... Well, because IsoTools uses the reference to reconstruct the transcriptome, and IsoQuant was run on a reference free mode. It is a bit of cheating to generate a transcriptome to annotate a genome using the reference ;). However, we will take advantage of the SQANTI3 Quality Control run we did with all of the orthogonal data (short-reads, CAGE peaks and polyA motifs), to obtain the best transcriptome posible. 

First, it is always a good practice to run SQANTI3 on the transcriptome, at least the Quality Control module, to get an idea of the state of the data we are working with. However, SQANTI3 needs a reference annotation to run. So... how can we use an annotation to generate an annotation? The solution is simple, we can create a "placebo" annotation, with one fake transcript in each chromosome in the fasta file, so SQANTI3 can run. The only thing we need to keep in mind is that many of the SQANTI3 classification columns will be useless, as no true reference annotation is given.

In [8]:
%%bash
conda run -n sqanti3 \
python3 generate_sqanti_placebo.py -i data/reference/GRCh38.p14.chr8.fa -o data/reference/sqanti_placebo.gtf

Placebo generated in data/reference/sqanti_placebo.gtf



</details><br>

Once the placebo annotation is created, we are free to run SQANTI3. (Skip this as we already did it in the preivous step)


In [10]:
%%bash
## DONT RUN, this data has been generated for you :)
conda run -n sqanti3 \
sqanti3_qc.py \
    --isoforms /repo/data/gge/VALT/isoquant/results/transcriptome_raw/OUT/OUT.transcript_models.gtf \
    --refGTF data/reference/sqanti_placebo.gtf \
    --refFasta data/reference/GRCh38.p14.chr8.fa \
    --coverage data/orthogonal/mapped/orthogonal_SJ.out.tab \
    --SR_bam data/orthogonal/mapped/orthogonal_Aligned.sortedByCoord.out.bam \
    --CAGE_peak data/orthogonal/human.refTSS_v3.1.hg38.bed \
    --polyA_motif data/orthogonal/mouse_and_human.polyA_motif.txt \
    --fl_count  /repo/data/gge/VALT/isoquant/results/transcriptome_raw/OUT/OUT.discovered_transcript_counts.tsv \
    --include_ORF --dir data/sqanti_output --output isoquant

mv results/qc/isoquant_corrected.cds.gff3 results/qc/isoquant_corrected.cds.gtf


      ░██████╗░░█████╗░
      ██╔═══██╗██╔══██╗
      ██║██╗██║██║░░╚═╝
      ╚██████╔╝██║░░██╗
      ░╚═██╔═╝░╚█████╔╝
      ░░░╚═╝░░░░╚════╝░
    
[INFO:2026-06-25 23:35:17,372] Write arguments to /repo/home/biouser2/notebooks/pablo/Annotation_practical/results/qc/isoquant.qc_params.txt
[INFO:2026-06-25 23:35:17,373] Initialising QC pipeline.
[INFO:2026-06-25 23:35:17,373] Parsing provided files
[INFO:2026-06-25 23:35:17,373] Reading genome fasta data/reference/GRCh38.p14.chr8.fa
[INFO:2026-06-25 23:35:18,293] **** Correcting sequences
[INFO:2026-06-25 23:35:18,293] Correcting fasta
[INFO:2026-06-25 23:35:18,293] Skipping aligning of sequences because GTF file was provided.
[INFO:2026-06-25 23:35:18,338] Indels will be not calculated since you ran SQANTI3 without alignment step (SQANTI3 with gtf format as transcriptome input).
[INFO:2026-06-25 23:35:18,599] **** Predicting ORF sequences...
[INFO:2026-06-25 23:35:18,599] Running TD2 ORF search on /repo/home/biouser2/notebooks/pablo/A

The most important step now is to propperly filter the annotation. As we saw before, the raw transcriptome that IsoTool (and the other transcript reconstruction methods) generate are not perfect and require some extra curation. In this case, we are faced with a unique circumstance. As we said before, we do not have a reference annotation to compare the isoforms agains (thus we used the placebo). This means, that there are many columns of the SQANTI classification that we usually use by default to filter, such as the distance_to_TSS or the structural_category, that are rendered useless. For this purpose, we have to be smart and use features of the classification file that are intrinsical to the isoforms, or belong to the orthogonal data used. For example, one thing to keep in mind is that we do not want isoforms with IntraPriming events, or flagged as possible Nonsense Mediated Decay. Other features that depend on the orthogonal data are the matches of the isoforms with the CAGE peaks or the support by short reads. However, keep in mind that we do not always have this information, depends on the experiment and species that we are using. 

**❓Trivia** Can you think of which columns, and their respective thresholds or values, that we could use for filtering?
<details><summary>Answer </summary>
    - No IntraPriming
    - PolyA motif found
    - No Nonsense Mediated Decay
    - No RTS stage
    - All junctions canonical OR with at least 3 short reads supporting
    - The TSS within a CAGE peak or with a ratio_TSS > 1.5
</details>

Try to create the annotation_filter_rules.json file on your own following the preivous directives. In case you get blocked, you can copy it from the data directory

In [11]:
%%bash 
conda run -n sqanti3 \
sqanti3_filter.py rules \
        --sqanti_class data/sqanti_output/isoquant_classification.txt \
        --filter_gtf  data/sqanti_output/isoquant_corrected.cds.gtf \
        --filter_isoforms data/sqanti_output/isoquant_corrected.fasta \
        --filter_faa  data/sqanti_output/isoquant_corrected.faa \
        --json_filter data/annotation_filter_rules.json \
        --dir results/evidence_based/sq_filter --output isoquant


      ███████╗██╗██╗░░░░░████████╗███████╗██████╗░
      ██╔════╝██║██║░░░░░╚══██╔══╝██╔════╝██╔══██╗
      █████╗░░██║██║░░░░░░░░██║░░░█████╗░░██████╔╝
      ██╔══╝░░██║██║░░░░░░░░██║░░░██╔══╝░░██╔══██╗
      ██║░░░░░██║███████╗░░░██║░░░███████╗██║░░██║
      ╚═╝░░░░░╚═╝╚══════╝░░░╚═╝░░░╚══════╝╚═╝░░╚═╝
    
[INFO:2026-06-25 23:38:59,237] Write arguments to /repo/home/biouser2/notebooks/pablo/Annotation_practical/results/evidence_based/sq_filter/isoquant_params.txt...
[INFO:2026-06-25 23:38:59,238] Running SQANTI3 filtering...
[INFO:2026-06-25 23:38:59,238] --------------------------------------------------
[INFO:2026-06-25 23:38:59,238]        Reading SQANTI3 classification file        
[INFO:2026-06-25 23:38:59,238] --------------------------------------------------
[INFO:2026-06-25 23:38:59,254] --------------------------------------------------
[INFO:2026-06-25 23:38:59,254]                 Reading JSON rules                
[INFO:2026-06-25 23:38:59,254] ------------------------

Once we have filtered the transcriptome, _voila!_ You have your genome annotation! Even thought this might look like the best and fastest option, is important to keep in mind that unless you have multiple tissues in the same sample, or from different conditions, you are only looking at a snapshot of the organisms genome. Rather than a genome annotation, you have a transcriptome annotaiton. However, it is a great start! :)



## 1.1 Quality Control

We will use three tools for the quality control of the annotation created with the transcriptome. These tools will be BUSCO, gffcompare and AGAT. 

1. [AGAT](https://github.com/NBISweden/AGAT) is a suite of scripts and modules that  has multiple functions, which can be obtained via `agat --tools`. In our case, it will help us extract the basic information about the annotation, such as the number of genes. 

2. [BUSCO](https://busco.ezlab.org/busco_userguide.html#protein-mode) run in protein mode will give a fast approximation of the number of core genes that were predicted, offering a measure of completeness to the annotation. 

3. [gffcompare](https://github.com/gpertea/gffcompare/) is a set of tools that allow the users to compare, merge or clean GFF files. This will be used to compare against a gold standard.

4. [OMARk](https://github.com/DessimozLab/OMArk) is a software similar to BUSCO, since it produces a quality assessment of the proteome based on the completeness and consitency. However, it has a twist, as it is able to detect contamination from closely related species, at the cost of longer runtimes than BUSCO. Thus, this one will not be run today. 

### Running AGAT

We will use two scripts from AGAT: `agat_convert_sp_gxf2gxf.pl` and `agat_sp_statistics.pl`. The first one will convert the Augustus output to GFF3 format, which is the standard format for an annotation, eliminating all the extra information that Augustus adds to the output. The second one will give us a summary of the annotation, including the number of genes, exons, introns and other features. 


In [12]:
%%bash
source activate augustus

agat_convert_sp_gxf2gxf.pl -g results/evidence_based/sq_filter/isoquant.filtered.gtf -o results/evidence_based/isoquant.agat_clean.gff
agat_sp_statistics.pl --gff  results/evidence_based/isoquant.agat_clean.gff -o results/quality_control/agat/evidence_based.stats


Duplicate specification "c|config=s" for option "c"
Duplicate specification "c|config=s" for option "config"



 ------------------------------------------------------------------------------
|   Another GFF Analysis Toolkit (AGAT) - Version: v1.4.1                      |
|   https://github.com/NBISweden/AGAT                                          |
|   National Bioinformatics Infrastructure Sweden (NBIS) - www.nbis.se         |
 ------------------------------------------------------------------------------
=> Using agat_config.yaml config file found in your working directory.
                                        
                                       
                          ------ Start parsing ------                           
-------------------------- parse options and metadata --------------------------
=> Accessing the feature_levels YAML file
Using standard /repo/home/biouser2/.conda/envs/augustus/lib/perl5/site_perl/auto/share/dist/AGAT/feature_levels.yaml file
=> Attribute used to group features when no Parent/ID relationship exists (i.e common tag):
	* locus_tag
	* gene_id
=>

Take a look at AGAT's statistics, as it will give you a summary of the annotation. **:question: Trivia: How many genes have been predicted?**
>TODO: Include the number of genes predicted
<details><summary>Solution</summary>
Include number
</details><br>

### Comparison againts a _gold standard_

In some cases, you will have access to a trusty annotation of your organism. There, it is a good practice to see how similar your annotation is to it. Since we are working on a human genome, we will do that today. We are going to use gffcompare, which will assess if the gene models predicted are similar to the real ones.

In [14]:
%%bash
mkdir -p results/quality_control/gffcompare
conda run -n augustus \
gffcompare -o results/quality_control/gffcompare/evidence_based -r data/reference/gencode.v45.chr8.gtf -T results/evidence_based/isoquant.agat_clean.gff

  10272 reference transcripts loaded.
  72 duplicate reference transcripts discarded.
  297 query transfrags loaded.



### Running BUSCO

**BUSCO** (Benchmarking Universal Single-Copy Orthologs), is a widely used tool designed to assess the completeness of genome assemblies and annotated gene sets. BUSCO identifies genes based on their evolutionary conservation as single-copy orthologs across specific phylogenetic branches. These conserved genes are expected to be present in all species within a lineage and typically exist as single copies. 

BUSCO offers multiple lineage-specific datasets tailored to different lineages of organisms. It is highly important to select the dataset that is the closest to our target species, to have as many genes as possible in the initial search. In order to check which datasets are available in BUSCO, and the one that is the most suitable for a given experiment, you can run the following command:

In [ ]:
%%bash
conda run -n augustus \
busco --list-dataset

In that long list you should find the best lineage to use for this annotation. I haven't filled that up yet hehe

**Try to do it on your own ;)**

In order to run BUSCO, you need to give as input the proteins, as it will compare the CDS of the genes in its high confidence set. For this annotation, SQANTI3 filter gave them to us directly, but later on, we will have to work to get them.

In [ ]:
%%bash
source activate augustus

busco -i results/evidence_based/sq_filter/isoquant.filtered.faa  -o results/quality_control/busco/evidence_based \
    -l primates_odb12 -m protein --miniprot \
    -c 3 --download_path /home/biouser2/course_data/annotation_practical/busco_downloads/


Since busco takes quite some time to run, the results have been precomputed for you. Take a look at them and assess how good (or bad) this annotation is.

# 2. _Ab initio_ annotation using AUGUSTUS

For the second annotation of the evening, we will ignore the extrinsic evidence and focus on what the genome and the mathematical models have to offer. We will use `augustus`, which is one of the most used tools for annotating genomes due to its flexibility. It allows the user to change many parameters, and it comes with pretty decent gene models that we can use in the most common organisms. This will be the case for today, as one of the most time consuming steps, due to the high manual curation, is the gene model creation. If you want to know more, you can check the extra notebook on gene model creation and training, but we won't cover it in this tutorial. 

In our case, we will select the human gene model precomputed by AUGUSTUS, anc carry out the _ab initio_ prediction of the annotation. This means that we will only use the gene model to predict the genes and create an structural annotation. Augustus _ab initio_ prediction works by employing a **generalized Hidden Markov Model**, statistically modelling the structure of genes within the genomic sequence, using only the provided gene model as reference.

<details>
<summary>Breakdown of the AUGUSTUS process</summary>

1. **States represent genomic features**: The GHMM defines different states that correspond to various parts of a gene and the surrounding genomic regions, such as exons, introns, start codons, stop codons, splice sites, and intergenic regions.

2. **Emission probabilities**: Each state in the model is associated with emission probabilities, which define the likelihood of that state producing a particular DNA sequence. For example, coding exon states will have probabilities that reflect the typical nucleotide composition of coding sequences.

3. **Transition probabilities**: The model also includes transition probabilities, which determine the likelihood of moving from one state to another in a biologically meaningful way. For instance, an intron state is likely to be followed by a splice site state.

4. **Finding the optimal parse**: Given an input DNA sequence, AUGUSTUS uses algorithms (similar to the Viterbi algorithm) to find the most probable sequence of states, also known as a parse, that could have generated that sequence. This optimal parse represents the predicted gene structure, identifying the locations and boundaries of genes, exons, and introns.

5. **Statistical modeling of gene components**: AUGUSTUS probabilistically models various characteristics of genes, including the sequence around splice sites, the branch point region, the start and stop codons, the length distribution of exons and introns, and even the distribution of the number of exons per gene. This allows the model to evaluate the likelihood of different potential gene structures based on learned statistical patterns from training data.

6. **No external evidence in ab initio prediction**: In its pure ab initio mode, AUGUSTUS relies solely on the information present within the input DNA sequence and the statistical models it has been trained on to predict gene structures.

Essentially, AUGUSTUS uses a sophisticated statistical framework to "decode" the genomic sequence and identify the most likely arrangement of gene components based on inherent sequence features and learned patterns.
</details><br>

This is the command to run augusuts. However, we won't be running this today, as it would take a long time to do (1.5 hours at least). The augusuts prediciton is not really resource heavy, but tends to consume time. One option to speed it up would be to parallelize a whole genome in chromosomes. Since this one chromosome, I have generated the result for you.


In [16]:
%%bash
## DO NOT RUN THIS, the output is there for you
conda run -n augustus \
augustus --species=human data/reference/GRCh38.p14.chr8.fa --protein=on --codingseq=on \
  --alternatives-from-sampling=true > results/ab_initio/Augustus_prediction.gff


CondaError: KeyboardInterrupt



TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType


We specify the options of `protein` and `codingseq`, so Augustus will include them as commentaries in the text and can be later extracted to generate the proteome for the quality control. 

❓**Trivia: How many genes have been predicted?**  
<details><summary>Solution</summary> 

</details>



## 2.1 Quality Control

Now that we know how these things work, we can breeze through them!

### Running AGAT

We will use two scripts from AGAT: `agat_convert_sp_gxf2gxf.pl` and `agat_sp_statistics.pl`. The first one will convert the Augustus output to GFF3 format, which is the standard format for an annotation, eliminating all the extra information that Augustus adds to the output. The second one will give us a summary of the annotation, including the number of genes, exons, introns and other features. 


In [15]:
%%bash
source activate augustus
mkdir -p results/quality_control/agat
agat_convert_sp_gxf2gxf.pl -g results/ab_initio/Augustus_prediction.gff -o results/ab_initio/Augustus_prediction.agat_clean.gff
agat_sp_statistics.pl --gff results/ab_initio/Augustus_prediction.agat_clean.gff -o results/quality_control/agat/ab_initio.stats

Duplicate specification "c|config=s" for option "c"
Duplicate specification "c|config=s" for option "config"



 ------------------------------------------------------------------------------
|   Another GFF Analysis Toolkit (AGAT) - Version: v1.4.1                      |
|   https://github.com/NBISweden/AGAT                                          |
|   National Bioinformatics Infrastructure Sweden (NBIS) - www.nbis.se         |
 ------------------------------------------------------------------------------
=> Using agat_config.yaml config file found in your working directory.
File results/ab_initio/Augustus_prediction.agat_clean.gff already exist.

 ------------------------------------------------------------------------------
|   Another GFF Analysis Toolkit (AGAT) - Version: v1.4.1                      |
|   https://github.com/NBISweden/AGAT                                          |
|   National Bioinformatics Infrastructure Sweden (NBIS) - www.nbis.se         |
 ------------------------------------------------------------------------------
=> Using agat_config.yaml config file found in y


Take a look at AGAT's statistics, as it will give you a summary of the annotation. **:question: Trivia: How many genes have been predicted?**
>TODO: Include the number of genes predicted
<details><summary>Solution</summary>
Include number
</details><br>

### Gold standard comparison

Same as before, but with a new annotation

In [28]:
%%bash
mkdir -p results/quality_control/gffcompare
conda run -n augustus \
gffcompare -o results/quality_control/gffcompare/ab_initio -r data/reference/gencode.v45.chr8.gtf \
    -T results/ab_initio/Augustus_prediction.agat_clean.gff

  10272 reference transcripts loaded.
  72 duplicate reference transcripts discarded.
  1423 query transfrags loaded.



### BUSCO analysis

Great! You got to the last step here. However, in order to do the busco assessment, we need to obtain the coding sequences associated with this annotation. Augustus is prepared for that, we just need to run one simple command.

In [31]:
%%bash
conda run -n augustus \
getAnnoFasta.pl results/ab_initio/Augustus_prediction.gff --seqfile=data/reference/GRCh38.p14.chr8.fa

Read in 1 sequence(s) from data/reference/GRCh38.p14.chr8.fa.



With the sequences in hand, its time to BUSCO!

In [32]:
%%bash
source activate augustus

busco -i results/ab_initio/Augustus_prediction.aa  -o results/quality_control/busco/ab_initio \
    -l primates_odb12 -m protein --miniprot \
    -c 3 --download_path /home/biouser2/course_data/annotation_practical/busco_downloads/


SyntaxError: invalid syntax (3360384260.py, line 1)

### Running OMARk

Running OMARk requires two steps (three if the database has not been downloaded). Similar to Augustus, you need to select the database that suits your genome the best. 

**:question: Trivia: Which database should we select?**
<details><summary>Solution</summary>
To be filled
</details><br>

OMARk relies on knowing where each protein in the query proteome maps within the precomputed gene families (HOGs) of the OMA database. To achieve this, [OMAmer](https://github.com/DessimozLab/omamer) first assigns proteins to HOGs using a fast, alignment-free k-mer-based method that compares the k-mer content of query proteins to those in the OMA database. This preprocessing step is essential for OMArk to analyze homologous relationships and taxonomic origins, enabling it to assess completeness, consistency, and detect contamination efficiently. For that reason, first we will run OMAmer before proceeding with OMARk.


In [ ]:
%%bash
# Download the database (optional)
wget <database_url> -O {output}
# OMAmer
omamer --db {input.omark_db} --query {input.proteome} --out {output}
omark -f {input.omamer} -d {input.omark_db} -o $(dirname {output})

# 3. _Evidence driven_ annotation

In this final section, lnc-RNA sequencing data will be integrated as part of the prediction step to improve the quality of the annotation. We will incorporate this data as hints, which will be used to guide AUGUSTUS to favor gene models consistent with the experimental data, resulting in more reliable structural and functional annotation, particularly for complex eukaryotic genomes and non-model organisms with limited existing genomic information. Pre-processing of lr-RNA seq data, such as transcript assembly and filtering, is often necessary to generate high-confidence evidence for AUGUSTUS.

Luckily, we have already done that, in the evidence based annotation! Yuhu, one less thing to do today :)


## 3.1 Hint creation

The final step of the transcriptome preprocessing is to generate a hints file, which is the format that Augustus will use to merge the lr-RNA evidence with the previous gene model. 


In [20]:
%%bash
source activate augustus
mkdir -p results/tmp
## Generate hints file
## We begin from the filtered transcriptome
gff=results/evidence_based/sq_filter/isoquant.filtered.gtf

grep -P "\t(CDS)\t" $gff | gtf2gff.pl --printIntron --out=results/tmp/tmp.gff

grep -P "\t(CDS|intron)\t" results/tmp/tmp.gff > results/tmp/tmp2.gff

# Remove gene_id and change transcript id for grp_id
sed -i 's/gene_id[^;]*;//g' results/tmp/tmp2.gff
sed -i 's/transcript_id \"/grp=/g' results/tmp/tmp2.gff

# change the trancript id for the source
sed "s/\";[[:space:]]*\r*$/;pri=1;src=lrRNA/g" results/tmp/tmp2.gff > results/evidence_driven/isquant.hints.gff

head results/evidence_driven/isquant.hints.gff


chr8	SQANTI3	CDS	457067	457100	.	-	.	grp=transcript18.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	intron	457101	494493	.	-	.	grp=transcript18.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	CDS	494494	494597	.	-	.	grp=transcript18.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	intron	494598	544649	.	-	.	grp=transcript18.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	CDS	544650	544868	.	-	.	grp=transcript18.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	CDS	664603	664676	.	-	.	grp=transcript40.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	intron	664677	668597	.	-	.	grp=transcript40.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	CDS	668598	668792	.	-	.	grp=transcript40.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	intron	668793	672021	.	-	.	grp=transcript40.chr8.nnic;pri=1;src=lrRNA
chr8	SQANTI3	CDS	672022	672204	.	-	.	grp=transcript40.chr8.nnic;pri=1;src=lrRNA



The hints file creation follows a complex process, which is described in the following steps:

<details>
<summary>Hint creation process</summary>

1. <strong>Filtering the input GTF file:</strong> The script filters the input GTF file to extract only the relevant features (CDS, exon, or intron) based on the specified UTR option. It uses <code>grep</code> to search for lines containing the specified feature types and then converts the GTF format to GFF format using <code>gtf2gff.pl</code>. The output is stored in a temporary directory.  

2. <strong>Removing gene_id and changing transcript_id:</strong> The script removes the <code>gene_id</code> field from the GFF file and replaces the <code>transcript_id</code> field with <code>grp_id</code>. This is done using <code>sed</code> to perform in-place text replacements.  

3. <strong>Adding source information:</strong> The script adds source information to the GFF file by replacing the <code>"</code> character with a custom string that includes the source (<code>PB</code>) and priority (<code>pri=1</code>). This is done using <code>sed</code> again.  

4. <strong>Outputting the final hints file:</strong> The final hints file is created by redirecting the modified GFF content to the specified output file. The temporary directory is removed afterward.  
</details> <br>


# 3.3 Final evidence-driven annotation

Now that we have the hints file, we can run Augustus again, but this time with the `--hints` flag. This will allow Augustus to use the hints file as a guide for the prediction. 


In [ ]:
%%bash
## DO NOT RUN
augustus --species=human da1ta/reference/GRCh38.p14.chr8.fa --protein=on --codingseq=on \
  --hintsfile=results/evidence_driven/isoquant.hints.gff \
  --extrinsicCfgFile=data/Augustus_config.cfg > results/evidence_driven/Augustus_isoquant.gff

The `--extrinsicCfgFile` parameter is used to specify the configuration file that contains the parameters for the hints. This file can be copied directly from the Augustus configuration directory and you should only add your source under the `[SOURCES]` section. The hints file will be used to guide the prediction, and the configuration file will specify how to use the hints. In our case, just copy the code below.

<details>
<summary>Configuration file</summary>

```ini
# extrinsic information configuration file for AUGUSTUS
# include with --extrinsicCfgFile=filename
# date: 01.08.2006
# Mario Stanke (mario@soe.ucsc.edu)


# source of extrinsic information:
# M manual anchor (required)
# P protein database hit
# E est database hit
# C combined est/protein database hit
# D Dialign
# R retroposed genes
# T transMapped refSeqs
# lrRNA PacBio (long reads, circular consensus)

[SOURCES]
M RM lrRNA

#
# individual_liability: Only unsatisfiable hints are disregarded. By default this flag is not set
# and the whole hint group is disregarded when one hint in it is unsatisfiable.
# 1group1gene: Try to predict a single gene that covers all hints of a given group. This is relevant for
# hint groups with gaps, e.g. when two ESTs, say 5' and 3', from the same clone align nearby.
#
[SOURCE-PARAMETERS]
lrRNA individual_liability
#   feature        bonus         malus   gradelevelcolumns
#               r+/r-
#
# the gradelevel colums have the following format for each source
# sourcecharacter numscoreclasses boundary    ...  boundary    gradequot  ...  gradequot
#

[GENERAL]
      start     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
       stop     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
        tss     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
        tts     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
        ass     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
        dss     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
   exonpart     1       0.98  M    1  1e+100  RM  1     1    lrRNA    1       1e5
       exon     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1e10
 intronpart     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1e5
     intron     1         .1  M    1  1e+100  RM  1     1    lrRNA    1       1e10
    CDSpart     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1e5
        CDS     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1e15
    UTRpart     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
        UTR     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
     irpart     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
nonexonpart     1          1  M    1  1e+100  RM  1  1.15    lrRNA    1       1
  genicpart     1          1  M    1  1e+100  RM  1     1    lrRNA    1       1
```

</details><br>

# 3.4 Quality control

The quality control of the final annotation is the same as before, but this time we will use the new annotation file as input for AGAT, gffcompare and BUSCO.

Try to do it in your own 😉

<details>
<summary>Click here if desperate (or lazy)</summary>


In [24]:
%%bash
source activate augustus

agat_convert_sp_gxf2gxf.pl -g results/evidence_driven/Augustus_isoquant.gff -o results/evidence_driven/Augustus_isoquant.agat_clean.gff
agat_sp_statistics.pl --gff results/evidence_driven/Augustus_isoquant.agat_clean.gff -o results/quality_control/agat/evidence_driven.stats

    #GFFcompare
gffcompare -o results/quality_control/gffcompare/evidence_driven -r data/reference/gencode.v45.chr8.gtf \
    -T results/evidence_driven/Augustus_isoquant.agat_clean.gff

# Proteome conversion
getAnnoFasta.pl results/evidence_driven/Augustus_isoquant.gff --seqfile=data/reference/GRCh38.p14.chr8.fa

Duplicate specification "c|config=s" for option "c"
Duplicate specification "c|config=s" for option "config"



 ------------------------------------------------------------------------------
|   Another GFF Analysis Toolkit (AGAT) - Version: v1.4.1                      |
|   https://github.com/NBISweden/AGAT                                          |
|   National Bioinformatics Infrastructure Sweden (NBIS) - www.nbis.se         |
 ------------------------------------------------------------------------------
=> Using agat_config.yaml config file found in your working directory.
                                        
                                       
                          ------ Start parsing ------                           
-------------------------- parse options and metadata --------------------------
=> Accessing the feature_levels YAML file
Using standard /repo/home/biouser2/.conda/envs/augustus/lib/perl5/site_perl/auto/share/dist/AGAT/feature_levels.yaml file
=> Attribute used to group features when no Parent/ID relationship exists (i.e common tag):
	* locus_tag
	* gene_id
=>

  10272 reference transcripts loaded.
  72 duplicate reference transcripts discarded.
  662 query transfrags loaded.


Read in 1 sequence(s) from data/reference/GRCh38.p14.chr8.fa.


In [ ]:
# BUSCO
busco -i results/evidence_driven/Augustus_isoquant.aa  -o results/quality_control/busco/evidence_driven \
    -l primates_odb12 -m protein --miniprot \
    -c 3 --download_path /home/biouser2/course_data/annotation_practical/busco_downloads/ 

# 4. Conclusion

With this, you have reached the end of this tutorial. You have learned how to create a gene model from scratch, how to use it to predict genes in a genome and how to use lr-RNA seq data to improve the prediction. You have also learned how to assess the quality of the annotation using different tools.

Now, compare the results of the three annotations. What are the main differences? Do you think that the lr-RNA seq data improved the prediction? Why?


In [32]:
import os
import glob
import json
import pandas as pd

# Define relative path from the notebook's directory
base_dir = "results/quality_control/busco"
run_types = ["evidence_based", "ab_initio", "evidence_driven","reference"]

data = []
for run in run_types:
    run_dir = os.path.join(base_dir, run)

    # 1. Check if the run directory exists
    if not os.path.isdir(run_dir):
        data.append({"Run Type": run, "Status": "Directory not found"})
        continue

    # 2. Look for the short summary JSON files recursively
    json_files = glob.glob(os.path.join(run_dir, "**", "*.json"), recursive=True)
    summary_files = [f for f in json_files if "short_summary" in os.path.basename(f)]

    # If the run is still in progress (no summary json yet)
    if not summary_files:
        data.append({"Run Type": run, "Status": "Running / No results yet"})
        continue

    # 3. Parse the results from the first summary JSON file
    try:
        with open(summary_files[0], "r") as f:
            summary = json.load(f)

        results = summary.get("results", {})
        lineage = summary.get("lineage_dataset", {}).get("name", "N/A")

        data.append({
            "Run Type": run,
            "Status": "Completed",
            "Lineage": lineage,
            "Complete (%)": results.get("Complete percentage"),
            "Complete": results.get("Complete BUSCOs"),
            "Single copy (%)": results.get("Single copy percentage"),
            "Single copy": results.get("Single copy BUSCOs"),
            "Multi copy (%)": results.get("Multi copy percentage"),
            "Multi copy": results.get("Multi copy BUSCOs"),
            "Fragmented (%)": results.get("Fragmented percentage"),
            "Fragmented": results.get("Fragmented BUSCOs"),
            "Missing (%)": results.get("Missing percentage"),
            "Missing": results.get("Missing BUSCOs"),
            "Total markers (n)": results.get("n_markers")
        })
    except Exception as e:
        data.append({"Run Type": run, "Status": f"Error: {str(e)}"})

# Create DataFrame
df = pd.DataFrame(data)

# Cast count columns to nullable integer type (Int64) to show them without decimals (.0)
count_cols = ["Complete", "Single copy", "Multi copy", "Fragmented", "Missing", "Total markers (n)"]
for col in count_cols:
    if col in df.columns:
        df[col] = df[col].astype("Int64")

# Return a styled HTML table in Jupyter (theme-agnostic styling)
df.style.format({
    "Complete (%)": "{:.1f}%",
    "Single copy (%)": "{:.1f}%",
    "Multi copy (%)": "{:.1f}%",
    "Fragmented (%)": "{:.1f}%",
    "Missing (%)": "{:.1f}%",
}, na_rep="-").set_properties(**{
    "text-align": "center"
}).set_table_styles([
    dict(selector="th", props=[
        ("text-align", "center"),
        ("font-weight", "bold"),
        ("background-color", "rgba(128, 128, 128, 0.15)"),  # Theme-aware light overlay
        ("padding", "8px"),
        ("border", "1px solid rgba(128, 128, 128, 0.3)")
    ]),
    dict(selector="td", props=[
        ("padding", "8px"),
        ("border", "1px solid rgba(128, 128, 128, 0.2)")
    ])
])

,Run Type,Status,Lineage,Complete (%),Complete,Single copy (%),Single copy,Multi copy (%),Multi copy,Fragmented (%),Fragmented,Missing (%),Missing,Total markers (n)
0,evidence_based,Completed,primates_odb12,1.1%,132,0.8%,99,0.3%,33,0.1%,6,98.8%,11696,11834
1,ab_initio,Completed,primates_odb12,2.5%,300,2.5%,300,0.0%,0,0.5%,56,97.0%,11478,11834
2,evidence_driven,Completed,primates_odb12,1.7%,201,1.5%,180,0.2%,21,0.3%,38,98.0%,11595,11834


In [31]:

import os
import re
import pandas as pd

# Define relative path from the notebook's directory
base_dir = "results/quality_control/gffcompare"
run_types = ["evidence_based", "ab_initio", "evidence_driven"]

data = []
for run in run_types:
    stats_file = os.path.join(base_dir, f"{run}.stats")

    # 1. Check if the stats file exists
    if not os.path.exists(stats_file):
        data.append({
            "Run Type": run,
            "Status": "File not found",
            "Query mRNAs": None, "Query Loci": None, "Matching Transcripts": None,
            "Base Sens (%)": None, "Base Prec (%)": None,
            "Exon Sens (%)": None, "Exon Prec (%)": None,
            "Intron Sens (%)": None, "Intron Prec (%)": None,
            "Intron chain Sens (%)": None, "Intron chain Prec (%)": None,
            "Transcript Sens (%)": None, "Transcript Prec (%)": None,
            "Locus Sens (%)": None, "Locus Prec (%)": None
        })
        continue

    # 2. Parse the stats file
    try:
        with open(stats_file, "r") as f:
            content = f.read()

        stats = {"Run Type": run, "Status": "Completed"}

        # Parse query details (e.g. Query mRNAs : 297 in 202 loci)
        query_match = re.search(r"Query mRNAs\s*:\s*(\d+)\s+in\s+(\d+)\s+loci", content)
        if query_match:
            stats["Query mRNAs"] = int(query_match.group(1))
            stats["Query Loci"] = int(query_match.group(2))

        # Parse matching count
        matching_match = re.search(r"Matching transcripts:\s*(\d+)", content)
        if matching_match:
            stats["Matching Transcripts"] = int(matching_match.group(1))

        # Parse Sensitivity & Precision levels
        levels = ["Base", "Exon", "Intron", "Intron chain", "Transcript", "Locus"]
        for level in levels:
            pattern = rf"{level}\s+level:\s*([\d\.-]+)\s*\|\s*([\d\.-]+)\s*\|"
            match = re.search(pattern, content)
            if match:
                sens = match.group(1).strip()
                prec = match.group(2).strip()
                stats[f"{level} Sens (%)"] = float(sens) if sens != "-" else None
                stats[f"{level} Prec (%)"] = float(prec) if prec != "-" else None

        data.append(stats)
    except Exception as e:
        data.append({
            "Run Type": run,
            "Status": f"Error: {str(e)}",
            "Query mRNAs": None, "Query Loci": None, "Matching Transcripts": None,
            "Base Sens (%)": None, "Base Prec (%)": None,
            "Exon Sens (%)": None, "Exon Prec (%)": None,
            "Intron Sens (%)": None, "Intron Prec (%)": None,
            "Intron chain Sens (%)": None, "Intron chain Prec (%)": None,
            "Transcript Sens (%)": None, "Transcript Prec (%)": None,
            "Locus Sens (%)": None, "Locus Prec (%)": None
        })

# Create DataFrame
df = pd.DataFrame(data)

# Cast count columns to nullable integer type Int64
count_cols = ["Query mRNAs", "Query Loci", "Matching Transcripts"]
for col in count_cols:
    if col in df.columns:
        df[col] = df[col].astype("Int64")

# Columns to format as percentages
pct_cols = [
    "Base Sens (%)", "Base Prec (%)",
    "Exon Sens (%)", "Exon Prec (%)",
    "Intron Sens (%)", "Intron Prec (%)",
    "Intron chain Sens (%)", "Intron chain Prec (%)",
    "Transcript Sens (%)", "Transcript Prec (%)",
    "Locus Sens (%)", "Locus Prec (%)"
]
format_dict = {col: "{:.1f}%" for col in pct_cols}

# Return a styled HTML table in Jupyter (theme-agnostic styling)
df.style.format(format_dict, na_rep="-").set_properties(**{
    "text-align": "center"
}).set_table_styles([
    dict(selector="th", props=[
        ("text-align", "center"),
        ("font-weight", "bold"),
        ("background-color", "rgba(128, 128, 128, 0.15)"),  # Theme-aware light overlay
        ("padding", "8px"),
        ("border", "1px solid rgba(128, 128, 128, 0.3)")
    ]),
    dict(selector="td", props=[
        ("padding", "8px"),
        ("border", "1px solid rgba(128, 128, 128, 0.2)")
    ])
])

,Run Type,Status,Query mRNAs,Query Loci,Matching Transcripts,Base Sens (%),Base Prec (%),Exon Sens (%),Exon Prec (%),Intron Sens (%),Intron Prec (%),Intron chain Sens (%),Intron chain Prec (%),Transcript Sens (%),Transcript Prec (%),Locus Sens (%),Locus Prec (%)
0,evidence_based,Completed,297,202,214,10.3%,95.2%,9.9%,95.8%,11.2%,98.7%,2.3%,72.8%,2.1%,72.1%,6.9%,84.2%
1,ab_initio,Completed,1423,1423,74,18.4%,67.9%,20.9%,49.7%,26.8%,49.0%,0.8%,6.5%,0.7%,5.2%,3.1%,5.2%
2,evidence_driven,Completed,662,588,156,10.7%,93.8%,11.8%,75.9%,15.4%,82.4%,1.7%,30.1%,1.5%,23.6%,5.6%,23.5%
